# Redes Neurais Artificiais Unidade 3 - Questões da Lista

## Questão 7 - Projeto Chatbot


Desenvolva um chatbot simples utilizando a biblioteca transformers da Hugging Face.

Requisitos mı́nimos:

- carregar um modelo pré-treinado;
- permitir interação;
- gerar respostas automaticamente;
- apresentar um pequeno relatório explicando o funcionamento do modelo utilizado.

---

Sugestão de bibliotecas:

- transformers
- torch

---

A entrega deve conter:
- código-fonte;
- prints da execução;
- descrição do modelo utilizado;
- análise dos resultados obtidos.

In [4]:
# Importando as bibliotecas
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from groq import Groq
from deep_translator import GoogleTranslator

In [ ]:
# Carregando o modelo pré-treinado + tokenizador
# Flan-T5-base: modelo encoder-decoder treinado para seguir instruções
#tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
#model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4614.86it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [6]:
#inicializando o cliente groq com a chamada da API
client = Groq(api_key="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")

In [7]:
# Prompt de sistema: definindo a persona e o domínio de geração do chatbot
SYSTEM_PROMPT = (
    "You are an expert book recommendation assistant."
    "Your only role is to recommend books based on the user's preferences."
    "Always suggest specific book titles, their authors, and briefly explain "
    "why each book matches the user's request."
    "Ask clarifying questions about genres or favorite authors when needed."
    "If the user doesn't specify a particular genre, recommend books that are well-reviewed by critics."
)

In [ ]:
# Criando uma função para o loop de bate-papo. Caso deseje encerrar a conversa com o Chatbot, digite "quit", "exit" ou "sair"
def run_chatbot():
    # lista de dicionários que armazena os turnos anteriores da conversa
    # 'role' segue o padrão da API: 'system', 'user' ou 'assistant'
    conversation_history = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]


    print("\n--- Inicializando o Chatbot ---")
    print("Digite 'quit', 'exit' ou 'sair' para encerrar a conversa.\n")

    # loop da conversa
    while True:
        user_input = input("Você: ")
        print(f"Você: {user_input}")

        # condicional para sair do loop (conversa)
        if user_input.lower() in ["quit","exit","sair"]:
            print("Chatbot: Tchau e até a próxima !")
            break

        # traduzindo a entrada do usuário de PT-BR para inglês antes de enviar ao modelo
        user_input_en = GoogleTranslator(source='pt', target='en').translate(user_input)

        # adicionando a nova entrada do usuário ao histórico da conversa
        conversation_history.append({"role": "user", "content": user_input_en})
        print(f"\n[Passo 1] Histórico enviado ao modelo: {len(conversation_history)} mensagens")

        # gerando uma resposta com a API da Groq
        print("[Passo 2] O modelo está gerando a resposta...")
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages= conversation_history,
            max_tokens= 200,
            temperature= 0.7,
        )

        # decodificando a resposta
        response_en = completion.choices[0].message.content

        # adicionando a resposta do modelo ao histórico da conversa
        conversation_history.append({"role": "assistant", "content": response_en})

        # traduzindo a resposta do modelo de inglês pra PT-BR
        response = GoogleTranslator(source='en', target='pt').translate(response_en)
        print(f"Chatbot: {response}")

# chamando a função para usar o chatbot
run_chatbot()


--- Inicializando o Chatbot ---
Digite 'quit', 'exit' ou 'sair' para encerrar a conversa.

Você: Olá

[Passo 1] Histórico enviado ao modelo: 2 mensagens
[Passo 2] O modelo está gerando a resposta...
Chatbot: Bem-vindo! Ficarei feliz em ajudá-lo a encontrar sua próxima ótima leitura. Você pode me dizer que tipo de livro você está com vontade? Você está interessado em ficção, não ficção, mistério, ficção científica, fantasia, romance ou outra coisa? Ou talvez você tenha um autor favorito que gostaria de explorar mais?
Você: Poderia me recomendar livros de fantasia dos últimos 20 anos ?

[Passo 1] Histórico enviado ao modelo: 4 mensagens
[Passo 2] O modelo está gerando a resposta...
Chatbot: Aqui estão alguns livros de fantasia altamente recomendados dos últimos 20 anos:

1. **O Nome do Vento** por Patrick Rothfuss (2007): Este romance de fantasia épica é o primeiro livro da série The Kingkiller Chronicle. É uma história lindamente escrita sobre um lendário músico e assassino, Kvothe, e 

### Relatório do Chatbot

Este relatório explica o funcionamento do modelo utilizado, com a descrição do modelo utilizado juntamente com a análise dos resultados obtidos. Vale mencionar que um passo a passo similar foi feito para a aplicação do *chatbot* em um ambiente de dashboard interativo, utilizando a biblioteca *streamlit*.

---

#### Modelo Utilizado: *DialogGPT-medium* 

O modelo que foi utilizado para a criação do chatbot **DialogGPT-medium**. Sendo sua arquitetura do modelo *transformer* do tipo *decoder-only*, ou seja, ele é focado na geração de texto, possuindo uma quantidade de aproximadamente 345 milhões de parâmetros na sua rede neural. 

Ele foi treinado pela microsoft exclusivamente para dialogos e o seu conjunto de dados utilizado para seu treinamento foram cerca de 170 milhões de conversas extraídas do *Reddit* e, por conta disso, o modelo tende a gerar respostas com o estilo informal e às vezes imprevisível dessa plataforma.

---

#### Importando as bibliotecas 

As bibliotecas que serão utilizadas para a aplicação serão a `transformers`, `torch`, `deep-translate`

##### `transformers`

Biblioteca utilizada para conseguir acesso aos modelos de arquitetura *Transformers* existentes na web em plataformas como *Hugging Face*. Ela será utilizada como ponte para conectar a nossa aplicação aos modelos de IA de código aberto pré-treinados.

- `AutoTokenizer`

Será utilizada para construir o nosso tokenizador (*tokenizer*), que será responsável por fatiar a frase em palavras ou pedaços de palavras, transformando os *tokens* para que o modelo consiga trabalhar em cima deles. Ele também fará o processo inverso: números -> tokenizador -> palavras.

- `AutoModelForCasualLM`

A modelagem de linguagem causal do modelo representa a capacidade matemática do modelo prever qual é a próxima palavra em uma sequência. Essa classe é passada para o modelo *model* para que seja carregado o "cérebro" do chatbot.

##### `torch`

Torch é utilizado para que seja possível realizar as manipulações matriciais com os tensores, representados por matrizes numéricas multidimensionais. 

##### `deep_translator`

O intuito desta biblioteca é traduzir as respostas geradas pelo modelo para português (PT-BR). Isso torna-se necessário uma vez que o modelo utilizado foi treinado com uma base de dados em inglês. Sem o tratamento prévio para português, o modelo ficaria muito confuso facilmente, tendendo a responder coisas sem sentido.

--- 

#### Como o *Chatbot* funciona

Este notebook apresenta uma célula de demonstração de execução do chatbot, definido pela função `run_chatbot()`. 

Inicialmente, como `chat_history_ids` é a variável que vai acumular o histórico da conversa em formato de tokens, ela começa com *None* pois ainda não há um histórico.

Ao entrar no loop *while*, enquanto o usuário não passar um *input* de "*quit*", "*exit*" ou "*sair*", o *loop* continuará a ser executado e o chat se comportará da seguinte forma:

1) O usuário realiza um entrada em português que é traduzida de pt-br para inglês por meio de `GoogleTranslator` (exportada da biblioteca `deep_translator`).

2) A entrada do usuário é codificada e é gerada uma máscara de atenção, para indicar aos tokens quais os outros tokens eles devem olhar com atenção para ganhar o contexto da sequencia de palavras. O intuito é converter o texto em inglês do usuário para uma lista de números que modelo entenda.

3) O histórico do contexto é atualizado por meio de uma concatenação com uma nova entrada do usuário ao histórico do chat, em um único tensor. `dim= -1` indica que a concatenação é feita na última dimensão do tensor (referente ao comprimento da sequência).

4) A mascara de atenção é atualizada a cada loop, pois a cada nova mensagem haverão novos tokens e esses novos tokens precisam saber a relevância das demais palavras da frase. 

5) Agora ocorre uma das partes mais importantes do fluxo de operação do *chatbot*: **a geração do texto utilizando o modelo pré-treinado**. É agora que modelo irá começar a gerar o texto, baseando-se sempre no cálculo da probabilidade do próximo token, tendo como base o mecanismo de atenção oriundo da arquitetura *Transformer*.

6) No final, a resposta gerada em inglês é decodificada para português e é devolvida como *response* para o usuário.

--- 

#### Resultados

Sem o uso de técnicas de re-treinamento (*fine tunning*), o modelo se comportou da seguinte forma:

- tempo de processamento

O tempo que levou para um *input* ser respondido variou pelo chat variou foi de aproximadamente 3 minutos (2:59 min) para prompts mais elaborados. 

- forma de resposta

Pelo fato da base de dados ser oriunda de *posts* do *Reddit*, a maioria das respostas que o *chatbot* entrega contem bastante ironia, como nos podemos ver nas imagens, logo abaixo (imagem 1) 